# Calculating Correlation in Climate Data

---

## Our dataset

We have 10 years of monthly average temperature (°C) and monthly total precipitation (mm) recorded at Millbrook weather station. We want to know: **do warmer months tend to be drier, or wetter?**

Let's enter the data and take a look at it.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Monthly average temperature (°C) and precipitation (mm) for Millbrook, 2014-2023
data = {
    'year':  [2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023],
    'temp_C': [22.1, 21.3, 23.8, 20.9, 24.2, 22.7, 25.1, 23.4, 26.0, 24.8],
    'precip_mm': [88, 102, 65, 110, 58, 79, 47, 71, 42, 61]
}

df = pd.DataFrame(data)
print(df.to_string(index=False))

---
## What is correlation?

Correlation measures **how strongly two variables move together**, and in what direction.

The result is a single number called **r**, which always falls between −1 and +1:

| r value | Meaning |
|---|---|
| Close to **+1** | As one variable increases, the other tends to increase too |
| Close to **−1** | As one variable increases, the other tends to decrease |
| Close to **0** | No clear relationship between the two variables |

The formula for the Pearson correlation coefficient is:

$$r = \frac{\sum (x_i - \bar{x})(y_i - \bar{y})}{\sqrt{\sum (x_i - \bar{x})^2 \cdot \sum (y_i - \bar{y})^2}}$$

This looks intimidating, but it's just measuring how much two variables deviate from their means *at the same time*. Let's calculate it step by step.

---
## Method 1: Calculate r manually

We will work directly with the DataFrame columns, following the formula above.

In [ ]:
# Step 1: Calculate the mean of each variable
mean_temp   = df['temp_C'].mean()
mean_precip = df['precip_mm'].mean()

print(f"Mean temperature:   {mean_temp:.2f} °C")
print(f"Mean precipitation: {mean_precip:.2f} mm")

In [ ]:
# Step 2: Calculate the deviations from the mean for each variable
df['temp_dev']   = df['temp_C']   - mean_temp
df['precip_dev'] = df['precip_mm'] - mean_precip

print(df[['year', 'temp_C', 'temp_dev', 'precip_mm', 'precip_dev']].to_string(index=False))

In [ ]:
# Step 3: Calculate the numerator and denominator of the r formula

# Numerator: sum of (temp deviation × precip deviation) for each year
numerator = (df['temp_dev'] * df['precip_dev']).sum()

# Denominator: sqrt( sum of squared temp deviations × sum of squared precip deviations )
denominator = np.sqrt((df['temp_dev']**2).sum() * (df['precip_dev']**2).sum())

# r
r_manual = numerator / denominator

print(f"Numerator:   {numerator:.2f}")
print(f"Denominator: {denominator:.2f}")
print(f"r (manual):  {r_manual:.3f}")

---
## Method 2: Using the pandas built-in `.corr()`

Pandas has a built-in method that calculates the correlation coefficient directly from a DataFrame. It should give the same answer as our manual calculation.

In [ ]:
# Pandas .corr() computes a full correlation matrix for all numeric columns
r_pandas = df['temp_C'].corr(df['precip_mm'])

print(f"r (pandas):  {r_pandas:.3f}")
print(f"r (manual):  {r_manual:.3f}")
print(f"\nDo they match? {np.isclose(r_manual, r_pandas)}")

---
## Plot: scatter plot of temperature vs. precipitation

A scatter plot is the standard way to visualize a correlation. Each point represents one year. If there is a relationship between the two variables, the points will form a pattern rather than a random cloud.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))

ax.scatter(df['temp_C'], df['precip_mm'], color='steelblue', s=60, zorder=3)

# Label each point with its year
for _, row in df.iterrows():
    ax.annotate(str(int(row['year'])),
                xy=(row['temp_C'], row['precip_mm']),
                xytext=(4, 4), textcoords='offset points', fontsize=8, color='gray')

ax.set_xlabel('Average Temperature (°C)')
ax.set_ylabel('Total Precipitation (mm)')
ax.set_title(f'Temperature vs. Precipitation — Millbrook\nr = {r_pandas:.3f}')
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---
## Interpreting the result

Our value of r is close to **−1**, which means there is a **strong negative correlation**: warmer years in Millbrook tend to be drier, and cooler years tend to be wetter.

You can see this in the scatter plot — the points slope downward from left to right.

> **⚠️ Important reminder:** A strong correlation does not mean one variable is *causing* the other to change. It only tells us they tend to move together. There may be a physical explanation, a third variable driving both, or the relationship may even be coincidental in a short dataset like this one. Always think critically about *why* a correlation might exist before drawing conclusions.

> **Written response:** Based on the scatter plot, describe the relationship between temperature and precipitation at Millbrook in your own words. Can you think of a physical reason why warmer months might also be drier ones? What would a climate scientist need to do before concluding this is a real, meaningful relationship?